# Tardis project

We, Loup, Lukas and Eva are part of a newly formed **SNCF Data Analysis Service**, dedicated to improving the efficiency of train travel across the country.

Our mission? Analyze historical train delay data, uncover hidden patterns, and develop a predictive model that can forecast delays before they happen. The SNCF has entrusted our team with making the railway system more efficient and transparent.

If we <span style="color:green">succeed</span>, our dashboard will be used by thousands of travelers to better plan their journeys. If we <span style="color:red">fail</span>... well, we expect a lot more unhappy commuters. 

No pressure! Using the provided dataset, our job is to clean and analyze historical delay data, develop a simple predictive model, and present our insights through an interactive **Streamlit dashboard**.

In [ ]:
#Importing the library to manage the dataset
import pandas as pd

# Step 1 : Data exploration and cleaning
## 1. Reading csv

Printing names of columns, number of columns and number of rows

In [ ]:
df = pd.read_csv("dataset.csv", sep=";")
print("Column names are: ")
#We get names of columns in the Dataframe
columns = df.columns
for i in range(len(columns) - 1):
    print(columns[i], end=", ")
print(columns[-1])

**Statistics on Dataframe per column**
- count : how many datas.
- unique : how many unique values.
- top : most common value.
- freq : most common value’s frequency.

In [ ]:
description = df.describe()
display(description)

#We print the dataframe
print("\nHere's the original dataset:")
display(df)

#We get number of line and columns in the Dataframe
nb_line_start, nb_col_start = df.shape
print(f"\nThere is {nb_col_start} columns and {nb_line_start} line in original dataset.")

## 2. Remove duplicate entries

In [ ]:
#This function return a Dataframe with the duplicates rows removed
df = df.drop_duplicates()

nb_line1, nb_col1 = df.shape
print(f"There is {nb_col1} columns and {nb_line1} line in the new dataset. {nb_line_start - nb_line1} lines has been removed.")

## 3. Cleaning dataset

### Date column
This column references the month and year observed for the trains.
1. Put date at correct format: YEAR-MONTH, remove spaces at the start and end of string and set correct hyphen

In [ ]:
#We get the values at column Date and put them to date format of pandas
df["Date"] = pd.to_datetime(df["Date"], yearfirst=True, format='mixed')

#Forcing format to be YEAR-MONTH
df["Date"] = df["Date"].dt.strftime("%Y-%m")

2. Check if the nearby dates are identical; if the current date does not have one, replace it, otherwise delete the line because it is too important data.

In [ ]:
values = df["Date"]

indexes = []
index_df = df.index
for i in index_df:
    value_str = str(values[i]).lower().strip()
    if value_str == "null" or value_str == "nan" or pd.isnull(values[i]):
        if i > 0 and i < len(values) - 1 and values[i - 1] == values[i + 1]:
            values[i] = values[i - 1]
        else:
            indexes.append(i)
df.update(values)

#Removing rows with dates unrectifable
df.drop(indexes, inplace=True)
nb_line2, nb_col2 = df.shape
print(f"There is {nb_col2} columns and {nb_line2} line in the new dataset. {nb_line1 - nb_line2} lines has been removed.")

### Service
If something else than "National" or "International", we put the value at NaN.

In [ ]:
values = df["Service"]
nb_modified = 0

for i in range(len(values)):
    value_str = str(values.iloc[i]).lower()
    if value_str != "national" and value_str != "international":
        values.iloc[i] = pd.NaT
        nb_modified += 1
df.update(values)

print(f"{nb_modified} cells has been modified.")

### Departure station

Delete every line without a departure station and puts them in uppercases

In [ ]:
values = df["Departure station"]

for i in range(len(values)):
    value_str = str(values.iloc[i]).upper()
    if value_str == "NAN" or value_str == "NULL":
        df.drop(i, inplace=True)

nb_line1, nb_col1 = df.shape
print(f"There is {nb_col1} columns and {nb_line1} line in the new dataset. {nb_line2 - nb_line1} lines has been removed.")

### Arrival station
Delete every line without a departure station and puts them in uppercases

In [ ]:
values = df["Arrival station"]

for i in range(len(values)):
    value_str = str(values.iloc[i]).upper()
    if value_str == "NAN" or value_str == "NULL":
        df.drop(i, inplace=True)

nb_line2, nb_col2 = df.shape
print(f"There is {nb_col2} columns and {nb_line2} line in the new dataset. {nb_line1 - nb_line2} lines has been removed.")

### Average journey time

Correct the format of the average journey time

1. Replace empty cells with -1 (-1 = No data, and will not be used for statistics)

In [ ]:
values = df["Average journey time"]
nb_modified = 0

for i in range(len(values)):
    if pd.isnull(values.iloc[i]):
        values.iloc[i] = "-1"
        nb_modified += 1

df.update(values)
print(f"{nb_modified} cells has been modified.")

2. Remove "min" from cells

In [ ]:
values = df["Average journey time"]
nb_modified = 0

for i in range(len(values)):
    if values.iloc[i].find("min") != -1:
        values.iloc[i] = values.iloc[i].replace("min", "")
        nb_modified += 1
    values.iloc[i] = values.iloc[i].strip()

df.update(values)
print(f"{nb_modified} cells has been modified.")

### Number of scheduled trains

Correct the format of the number of scheduled trains

Replace empty cells with -1 (-1 = No data, and will not be used for statistics)

In [ ]:
values = df["Number of scheduled trains"]
nb_modified = 0

for i in range(len(values)):
    if pd.isnull(values.iloc[i]):
        values.iloc[i] = "-1"
        nb_modified += 1

df.update(values)
print(f"{nb_modified} cells has been modified.")

### Number of cancelled trains

Correct the format of the number of cancelled trains

Replace empty cells with -1 (-1 = No data, and will not be used for statistics)

In [ ]:
values = df["Number of cancelled trains"]
nb_modified = 0

for i in range(len(values)):
    if pd.isnull(values.iloc[i]):
        values.iloc[i] = "-1"
        nb_modified += 1

df.update(values)
print(f"{nb_modified} cells has been modified.")

### Cancellation comments

Cancellation comments removed because there is no data inside it

In [ ]:
df = df.drop("Cancellation comments", axis=1)

print("New column names are: ")
#We get names of columns in the Dataframe
columns = df.columns
for i in range(len(columns) - 1):
    print(columns[i], end=", ")
print(columns[-1])

### Number of trains delayed at departure

Correct the format of the number of trains delayed at departure

Replace empty cells with -1 (-1 = No data, and will not be used for statistics)



In [ ]:
values = df["Number of trains delayed at departure"]
nb_modified = 0

for i in range(len(values)):
    if pd.isnull(values.iloc[i]):
        values.iloc[i] = "-1"
        nb_modified += 1

df.update(values)
print(f"{nb_modified} cells has been modified.")

### Average delay of late trains at departure

Correct the format of the average delay of late trains at departure

1. Replace empty cells with -1 (-1 = No data, and will not be used for statistics)

In [ ]:
values = df["Average delay of late trains at departure"]
nb_modified = 0

for i in range(len(values)):
    if pd.isnull(values.iloc[i]):
        values.iloc[i] = "-1"
        nb_modified += 1

df.update(values)
print(f"{nb_modified} cells has been modified.")

2. Remove "min" from cells

In [ ]:
values = df["Average delay of late trains at departure"]
nb_modified = 0

for i in range(len(values)):
    if values.iloc[i].find("min") != -1:
        values.iloc[i] = values.iloc[i].replace("min", "")
        nb_modified += 1
    values.iloc[i] = values.iloc[i].strip()

df.update(values)
print(f"{nb_modified} cells has been modified.")

### Average delay of all trains at departure

Correct the format of the number of the average delay of all trains at departure

1. Replace empty cells with -1 (-1 = No data, and will not be used for statistics)

In [ ]:
values = df["Average delay of all trains at departure"]
nb_modified = 0

for i in range(len(values)):
    if pd.isnull(values.iloc[i]):
        values.iloc[i] = "-1"
        nb_modified += 1

df.update(values)
print(f"{nb_modified} cells has been modified.")

2. Remove "min" from cells

In [ ]:
values = df["Average delay of all trains at departure"]
nb_modified = 0

for i in range(len(values)):
    if values.iloc[i].find("min") != -1:
        values.iloc[i] = values.iloc[i].replace("min", "")
        nb_modified += 1
    values.iloc[i] = values.iloc[i].strip()

df.update(values)
print(f"{nb_modified} cells has been modified.")

### Departure delay comments

Departure delay comments removed because there is no data inside it

In [ ]:
df = df.drop("Departure delay comments", axis=1)

print("New column names are: ")
#We get names of columns in the Dataframe
columns = df.columns
for i in range(len(columns) - 1):
    print(columns[i], end=", ")
print(columns[-1])

### Number of trains delayed at arrival

Correct the format of the number of the number of trains delayed at arrival

Replace empty cells with -1 (-1 = No data, and will not be used for statistics)

In [ ]:
values = df["Number of trains delayed at arrival"]
nb_modified = 0

for i in range(len(values)):
    if pd.isnull(values.iloc[i]):
        values.iloc[i] = "-1"
        nb_modified += 1

df.update(values)
print(f"{nb_modified} cells has been modified.")

### Average delay of late trains at arrival

Correct the format of the average delay of late trains at arrival

1. Replace empty cells with -1 (-1 = No data, and will not be used for statistics)

In [ ]:
values = df["Average delay of late trains at arrival"]
nb_modified = 0

for i in range(len(values)):
    if pd.isnull(values.iloc[i]):
        values.iloc[i] = "-1"
        nb_modified += 1

df.update(values)
print(f"{nb_modified} cells has been modified.")

2. Remove "min" from cells

In [ ]:
values = df["Average delay of late trains at arrival"]
nb_modified = 0

for i in range(len(values)):
    if values.iloc[i].find("min") != -1:
        values.iloc[i] = values.iloc[i].replace("min", "")
        nb_modified += 1
    values.iloc[i] = values.iloc[i].strip()

df.update(values)
print(f"{nb_modified} cells has been modified.")

### Average delay of all trains at arrival

Correct the format of the average delay of all trains at arrival

1. Replace empty cells with -1 (-1 = No data, and will not be used for statistics)


In [ ]:
values = df["Average delay of all trains at arrival"]
nb_modified = 0

for i in range(len(values)):
    if pd.isnull(values.iloc[i]):
        values.iloc[i] = "-1"
        nb_modified += 1

df.update(values)
print(f"{nb_modified} cells has been modified.")

2. Remove "min" from cells

In [ ]:
values = df["Average delay of all trains at arrival"]
nb_modified = 0

for i in range(len(values)):
    if values.iloc[i].find("min") != -1:
        values.iloc[i] = values.iloc[i].replace("min", "")
        nb_modified += 1
    values.iloc[i] = values.iloc[i].strip()

df.update(values)
print(f"{nb_modified} cells has been modified.")

### Arrival delay comments

Change every empty cells or cells without comments (NC, NA Non communiqué...)

In [ ]:
values = df["Arrival delay comments"]
nb_modified = 0

for i in range(len(values)):
    if pd.isnull(values.iloc[i]) or str(values.iloc[i]).lower().strip() in ["nc", "na", "non communiqué", "null", "nan"]:
        values.iloc[i] = "NC"
        nb_modified += 1

df.update(values)
print(f"{nb_modified} cells has been modified.")

### Number of trains delayed > 15min

Correct the format of the number of trains delayed > 15min

1. Replace empty cells with -1 (-1 = No data, and will not be used for statistics)

In [ ]:
values = df["Number of trains delayed > 15min"]
nb_modified = 0

for i in range(len(values)):
    if pd.isnull(values.iloc[i]):
        values.iloc[i] = "-1"
        nb_modified += 1

df.update(values)
print(f"{nb_modified} cells has been modified.")

2. Clean the number

In [ ]:
values = df["Number of trains delayed > 15min"]

for i in range(len(values)):
    values.iloc[i] = values.iloc[i].strip()

df.update(values)

### Average delay of trains > 15min (if competing with flights)

Correct the format of the average delay of trains > 15min (if competing with flights)

1. Replace empty cells with -1 (-1 = No data, and will not be used for statistics)

In [ ]:
values = df["Average delay of trains > 15min (if competing with flights)"]
nb_modified = 0

for i in range(len(values)):
    if pd.isnull(values.iloc[i]):
        values.iloc[i] = "-1"
        nb_modified += 1

df.update(values)
print(f"{nb_modified} cells has been modified.")

2. Clean the number

In [ ]:
values = df["Average delay of trains > 15min (if competing with flights)"]

for i in range(len(values)):
    values.iloc[i] = values.iloc[i].strip()

df.update(values)

### Number of trains delayed > 30min

Correct the format of the number of trains delayed > 30min

1. Replace empty cells with -1 (-1 = No data, and will not be used for statistics)

In [ ]:
values = df["Number of trains delayed > 30min"]
nb_modified = 0

for i in range(len(values)):
    if pd.isnull(values.iloc[i]):
        values.iloc[i] = "-1"
        nb_modified += 1

df.update(values)
print(f"{nb_modified} cells has been modified.")

2. Clean the number

In [ ]:
values = df["Number of trains delayed > 30min"]

for i in range(len(values)):
    values.iloc[i] = values.iloc[i].strip()

df.update(values)

### Number of trains delayed > 60min

Correct the format of the number of trains delayed > 60min

1. Replace empty cells with -1 (-1 = No data, and will not be used for statistics)

In [ ]:
values = df["Number of trains delayed > 60min"]
nb_modified = 0

for i in range(len(values)):
    if pd.isnull(values.iloc[i]):
        values.iloc[i] = "-1"
        nb_modified += 1

df.update(values)
print(f"{nb_modified} cells has been modified.")

2. Clean the number

In [ ]:
values = df["Number of trains delayed > 60min"]

for i in range(len(values)):
    values.iloc[i] = values.iloc[i].strip()

df.update(values)

### Pct delay due to external causes

Correct the format of the Pct delay due to external causes

1. Replace empty cells with -1 (-1 = No data, and will not be used for statistics)

In [ ]:
values = df["Pct delay due to external causes"]
nb_modified = 0

for i in range(len(values)):
    if pd.isnull(values.iloc[i]):
        values.iloc[i] = "-1"
        nb_modified += 1

df.update(values)
print(f"{nb_modified} cells has been modified.")

2. Clean the number

In [ ]:
values = df["Pct delay due to external causes"]

for i in range(len(values)):
    values.iloc[i] = values.iloc[i].strip()

df.update(values)

### Pct delay due to infrastructure

Correct the format of the pct delay due to infrastructure

1. Replace empty cells with -1 (-1 = No data, and will not be used for statistics)

In [ ]:
values = df["Pct delay due to infrastructure"]
nb_modified = 0

for i in range(len(values)):
    if pd.isnull(values.iloc[i]):
        values.iloc[i] = "-1"
        nb_modified += 1

df.update(values)
print(f"{nb_modified} cells has been modified.")

2. Clean the number

In [ ]:
values = df["Pct delay due to infrastructure"]

for i in range(len(values)):
    values.iloc[i] = values.iloc[i].strip()

df.update(values)

### Pct delay due to traffic management

Correct the format of the pct delay due to traffic management

1. Replace empty cells with -1 (-1 = No data, and will not be used for statistics)

In [ ]:
values = df["Pct delay due to traffic management"]
nb_modified = 0

for i in range(len(values)):
    if pd.isnull(values.iloc[i]):
        values.iloc[i] = "-1"
        nb_modified += 1

df.update(values)
print(f"{nb_modified} cells has been modified.")

2. Clean the number

In [ ]:
values = df["Pct delay due to traffic management"]

for i in range(len(values)):
    values.iloc[i] = values.iloc[i].strip()

df.update(values)

### Pct delay due to rolling stock

Correct the format of the pct delay due to rolling stock

1. Replace empty cells with -1 (-1 = No data, and will not be used for statistics)

In [ ]:
values = df["Pct delay due to rolling stock"]
nb_modified = 0

for i in range(len(values)):
    if pd.isnull(values.iloc[i]):
        values.iloc[i] = "-1"
        nb_modified += 1

df.update(values)
print(f"{nb_modified} cells has been modified.")

2. Clean the number

In [ ]:
values = df["Pct delay due to rolling stock"]

for i in range(len(values)):
    values.iloc[i] = values.iloc[i].strip()

df.update(values)

### Pct delay due to station management and equipment reuse

Correct the format of the pct delay due to station management and equipment reuse

1. Replace empty cells with -1 (-1 = No data, and will not be used for statistics)


In [ ]:
values = df["Pct delay due to station management and equipment reuse"]
nb_modified = 0

for i in range(len(values)):
    if pd.isnull(values.iloc[i]):
        values.iloc[i] = "-1"
        nb_modified += 1

df.update(values)
print(f"{nb_modified} cells has been modified.")

2. Clean the number

In [ ]:
values = df["Pct delay due to station management and equipment reuse"]

for i in range(len(values)):
    values.iloc[i] = values.iloc[i].strip()

df.update(values)

### Pct delay due to passenger handling (crowding, disabled persons, connections)

Correct the format of the pct delay due to passenger handling (crowding, disabled persons, connections)

1. Replace empty cells with -1 (-1 = No data, and will not be used for statistics)

In [ ]:
values = df["Pct delay due to passenger handling (crowding, disabled persons, connections)"]
nb_modified = 0

for i in range(len(values)):
    if pd.isnull(values.iloc[i]):
        values.iloc[i] = "-1"
        nb_modified += 1

df.update(values)
print(f"{nb_modified} cells has been modified.")

2. Clean the number

In [ ]:
values = df["Pct delay due to passenger handling (crowding, disabled persons, connections)"]

for i in range(len(values)):
    values.iloc[i] = values.iloc[i].strip()

df.update(values)

## Dataset after cleaning

In [ ]:

nb_line, nb_col = df.shape

print(f"\nThere is {nb_col} columns and {nb_line} line in the new dataset.\n- {nb_line_start - nb_line} lines has been removed\n- {nb_col_start - nb_col} columns has been removed.\n {((nb_line_start - nb_line) / nb_line_start * 100):.2f}% of the dataset lines were removed")

display(df)

Exporting the dataset cleaned

In [ ]:
df.to_csv("cleaned_dataset.csv", sep=";", index=False)